In [ ]:
import os
library_path = os.path.abspath(os.path.join(os.getcwd(), '../../molass-library'))
legacy_path = os.path.abspath(os.path.join(os.getcwd(), '../../molass-legacy'))
python_path = os.pathsep.join([library_path, legacy_path])
os.environ['PYTHONPATH'] = python_path
python_path

In [ ]:

import sys
sys.path.insert(0, '../../regals/python')
sys.path.insert(0, library_path)
sys.path.insert(0, legacy_path)
from molass import get_version
assert get_version(toml_only=True) >= '0.8.0', "Please update molass to version 0.8.0 or higher."
from molass_legacy import get_version as get_legacy_version
assert get_legacy_version(toml_only=True) >= '0.5.0', "Please update molass-legacy to version 0.5.0 or higher."
import numpy as np
import matplotlib.pyplot as plt
import h5py

In [ ]:
CypA_Tjump = {}
PATH1 = '../../regals/demo/data/CypA_Tjump.mat'
PATH3 = '../../regals/demo/data/NrdE_mix_AEX.mat'
data_path = PATH3
with h5py.File(data_path, 'r') as f:
    for key in f:
        CypA_Tjump[key] = np.array(f[key])

#Store raw data
q = CypA_Tjump['q'][0]
I = np.flip(CypA_Tjump['I'].T[:,1:], axis=1)
sigma = np.flip(CypA_Tjump['sigma'].T[:,1:], axis=1)
# delay_ns = CypA_Tjump['delay_ns'][0][1:]

In [ ]:
q.shape, I.shape, sigma.shape

In [ ]:
plt.plot(q, I[:,0])
plt.xlabel('q (1/A)')
plt.ylabel('Intensity (a.u.)')
plt.title('Scattering Intensity vs q')
plt.show()

In [ ]:
from molass.DataObjects.SecSaxsData import SecSaxsData as SSD
from molass.DataObjects.XrData import XrData
jv = np.arange(I.shape[1])
xr_data = XrData(q, jv, I, sigma)
ssd = SSD(object_list=[xr_data, None])
ssd.plot_3d()


In [ ]:
trimmed_ssd = ssd.trimmed_copy()
corrected_copy = trimmed_ssd.corrected_copy()

In [ ]:
rgcurve = corrected_copy.xr.compute_rgcurve()

In [ ]:
decomposition = corrected_copy.quick_decomposition(num_components=5)
decomposition.plot_components(title="EGH Decomposition", rgcurve=rgcurve)

In [ ]:
decomposition_sdm = decomposition.optimize_with_model('SDM', debug=True)
decomposition_sdm.plot_components(title="SDM Decomposition", rgcurve=rgcurve)

In [ ]:
runinfo = decomposition.optimize_rigorously(rgcurve=rgcurve, analysis_folder='temp_regals_comparison', method='NS')

In [ ]:
# rom molass_legacy.Optimizer.MplMonitor import MplMonitor
# MplMonitor.cleanup_orphaned_processes()

In [ ]:
current_decomposition = runinfo.get_current_decomposition(wait_for_first_results=True, debug=True)
current_decomposition.plot_components(rgcurve=rgcurve)